# Part d
### Testing different activation functions

In [18]:
from pathlib import Path
import sys


here = Path.cwd()
candidates = [here] + list(here.parents)
for p in candidates:
    if (p / "Code").is_dir():
        sys.path.insert(0, str(p))
        break
else:
    raise RuntimeError("Couldn't find a 'Code' folder in this project.")



import autograd.numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from Code.ffnn2 import NeuralNetwork as FFNN 
from Code.scheduler import Adam, RMS_prop, Constant, Scheduler
from Code.data import runge_function, make_data
from Code.cost import CostOLS, dCostOLS
from Code.comparison import run_comparison
from Code.plot import plot_sweep_heatmap, plot_activation_sweep_heatmap
from Code.activations import sigmoid, RELU, LRELU, identity, derivate
from Code.scheduler import Constant, RMS_prop, Adam
from Code.architectures import build_architectures, build_architectures_exact
from Code.helpers import build_nn, make_sched, train_eval_once, sweep, top_k, best_per
%matplotlib inline


import pandas as pd
import seaborn as sns 
sns.set_theme(style="white", font_scale=1.3)
plt.rcParams["figure.figsize"] = (14, 8)
plt.rcParams["figure.dpi"] = 200
plt.rcParams.update({
    "savefig.dpi": 200,
    "savefig.bbox": "tight"
})

rho_val, rho2_val = 0.9, 0.999
optimizers_to_sweep = {
    'GD':       (Constant, {}),
    'SGD':      (Constant, {}),      
    'RMS_prop': (RMS_prop, {'rho': rho_val}),
    'Adam':     (Adam,     {'rho': rho_val, 'rho2': rho2_val}),
}


In [19]:
#Initialize data

(seed_fixed,
 rng_fixed,
 X_fixed_test,
 y_fixed_test,
 N,
 rng,
 x,
 noise,
 y,
 X_train,
 X_val,
 y_train,
 y_val,
 scaler,
 X_train_scaled,
 X_val_scaled,
 X_fixed_test_scaled) = make_data(seed_fixed=42, N=200, noise_std=0.1, test_N=2000)



In [20]:


input_nodes = X_train_scaled.shape[1] # Features from scaled data
output_nodes = 1
layer_output_sizes_2H = [50, 100, output_nodes]
n_hidden_layers = len(layer_output_sizes_2H) - 1  # Should be 2 hidden layers
eta_constant = 1e-2 # if fixed


activation_tests = {
    'Sigmoid': sigmoid,
    'RELU': RELU,
    'LRELU': LRELU
}

epochs_sweep = 100 #keep low for speed, retrain later with top k performers 
batches = 128
lam = 0.0
eta_vals = np.logspace(-4, -1, 4)  # less values than before for speed (removed the worst performing ones)


architectures_to_sweep = build_architectures_exact(out_dim=output_nodes)

## Rough sweep over hyperparameters, activations, architecture, eta, gather the best MSE results

In [ ]:

try:
    eta_values = list(eta_constant)   # iterate over several values
except TypeError:
    eta_values = [eta_constant]       # single value

all_activation_results = {}
all_architectural_results = {}
all_results = {}

for arch_name, layer_output_sizes in architectures_to_sweep.items():
    n_hidden_layers = len(layer_output_sizes) - 1
    print(f"\n================ Architecture: {arch_name} ================")
    arch_bucket = {}

    for act_name, h_func in activation_tests.items():
        # hidden activations + identity output (regression)
        activation_funcs = [h_func]*n_hidden_layers + [identity]
        activation_ders  = [derivate(f) for f in activation_funcs]

        act_bucket = {}

        for opt_name, (optimizer_class, fixed_params) in optimizers_to_sweep.items():
            per_eta = {}

            for eta in eta_vals:
                current_kwargs = {'eta': eta}
                current_kwargs.update(fixed_params)

                
                batches_arg = 1 if opt_name == 'GD' else batches

                # train
                nn = FFNN(
                    network_input_size=input_nodes,
                    layer_output_sizes=tuple(layer_output_sizes),
                    activation_funcs=activation_funcs,
                    activation_ders=activation_ders,
                    cost_fun=CostOLS,
                    cost_der=dCostOLS,
                    seed=42
                )
                nn.reset_weights()

                scheduler_instance = optimizer_class(**current_kwargs)

                scores = nn.fit(
                    X_train_scaled, y_train,
                    scheduler=scheduler_instance,
                    batches=batches_arg,               
                    epochs=epochs_sweep,
                    lam=lam
                )

                # Evaluate on the fixed test set
                y_pred = nn.predict(X_fixed_test_scaled)
                test_mse = CostOLS(y_pred.ravel(), y_fixed_test.ravel())

                # Depth/width summary for the print
                depth = max(0, len(layer_output_sizes) - 1)
                width = (layer_output_sizes[0] if depth > 0 else 0)

                print(f" {act_name} + {opt_name} | eta={eta:.1e} | depth={depth}, width={width}: Test MSE={test_mse:.6f}")
                per_eta[float(eta)] = float(test_mse)

            # If only one eta was provided, keep your original shape: opt -> mse
            # If multiple etas, store a dict: opt -> {eta -> mse}
            # Always store per-eta dict, even if it has length 1
            act_bucket[opt_name] = per_eta

        # store results per activation
        arch_bucket[act_name] = act_bucket

    # store results per architecture
    all_results[arch_name] = arch_bucket 
    all_architectural_results[arch_name] = arch_bucket 
    # acts = nn._feed_forward_saver(X_train_scaled)[0] 




================ Architecture: 0_Hidden_Layers ================
 Sigmoid + GD | eta=1.0e-04 | depth=0, width=0: Test MSE=0.167003
 Sigmoid + GD | eta=1.0e-03 | depth=0, width=0: Test MSE=0.146659
 Sigmoid + GD | eta=1.0e-02 | depth=0, width=0: Test MSE=0.097361
 Sigmoid + GD | eta=1.0e-01 | depth=0, width=0: Test MSE=0.094747
 Sigmoid + SGD | eta=1.0e-04 | depth=0, width=0: Test MSE=0.101398
 Sigmoid + SGD | eta=1.0e-03 | depth=0, width=0: Test MSE=0.094750
 Sigmoid + SGD | eta=1.0e-02 | depth=0, width=0: Test MSE=0.095434
 Sigmoid + SGD | eta=1.0e-01 | depth=0, width=0: Test MSE=0.096421
 Sigmoid + RMS_prop | eta=1.0e-04 | depth=0, width=0: Test MSE=0.096012
 Sigmoid + RMS_prop | eta=1.0e-03 | depth=0, width=0: Test MSE=0.095160
 Sigmoid + RMS_prop | eta=1.0e-02 | depth=0, width=0: Test MSE=0.097823
 Sigmoid + RMS_prop | eta=1.0e-01 | depth=0, width=0: Test MSE=0.111308
 Sigmoid + Adam | eta=1.0e-04 | depth=0, width=0: Test MSE=0.108946
 Sigmoid + Adam | eta=1.0e-03 | depth=0, width=

/Users/selmabeateovland/Documents/Fys-stk4155/Project2/Project_2/.venv/lib/python3.12/site-packages/autograd/numpy/numpy_vjps.py:53: RuntimeWarning: overflow encountered in square
  lambda ans, x, y : unbroadcast_f(y, lambda g: - g * x / y**2))
/Users/selmabeateovland/Documents/Fys-stk4155/Project2/Project_2/.venv/lib/python3.12/site-packages/autograd/tracer.py:48: RuntimeWarning: overflow encountered in exp
  return f_raw(*args, **kwargs)
/Users/selmabeateovland/Documents/Fys-stk4155/Project2/Project_2/.venv/lib/python3.12/site-packages/autograd/numpy/numpy_vjps.py:75: RuntimeWarning: invalid value encountered in multiply
  defvjp(anp.exp,    lambda ans, x : lambda g: ans * g)


 Sigmoid + SGD | eta=1.0e-01 | depth=1, width=100: Test MSE=nan
 Sigmoid + RMS_prop | eta=1.0e-04 | depth=1, width=100: Test MSE=0.094602
 Sigmoid + RMS_prop | eta=1.0e-03 | depth=1, width=100: Test MSE=0.100625
 Sigmoid + RMS_prop | eta=1.0e-02 | depth=1, width=100: Test MSE=0.094430
 Sigmoid + RMS_prop | eta=1.0e-01 | depth=1, width=100: Test MSE=0.123721
 Sigmoid + Adam | eta=1.0e-04 | depth=1, width=100: Test MSE=0.094285
 Sigmoid + Adam | eta=1.0e-03 | depth=1, width=100: Test MSE=0.095130
 Sigmoid + Adam | eta=1.0e-02 | depth=1, width=100: Test MSE=0.022198
 Sigmoid + Adam | eta=1.0e-01 | depth=1, width=100: Test MSE=0.033559
 RELU + GD | eta=1.0e-04 | depth=1, width=100: Test MSE=0.167068
 RELU + GD | eta=1.0e-03 | depth=1, width=100: Test MSE=0.146614
 RELU + GD | eta=1.0e-02 | depth=1, width=100: Test MSE=0.096790
 RELU + GD | eta=1.0e-01 | depth=1, width=100: Test MSE=0.090351
 RELU + SGD | eta=1.0e-04 | depth=1, width=100: Test MSE=0.100959
 RELU + SGD | eta=1.0e-03 | depth=

## Find best result

In [ ]:
rows = []

for arch_name, act_dict in all_results.items():
    sizes = architectures_to_sweep.get(arch_name)
    depth = (len(sizes) - 1) if sizes is not None else None
    width = (sizes[0] if sizes and depth and depth > 0 else None)

    for act_name, opt_dict in act_dict.items():
        for opt_name, per_eta in opt_dict.items():
            # normalize in case any old entries are floats
            if not isinstance(per_eta, dict):
                per_eta = {float(eta_constant): float(per_eta)}

            for eta, mse in per_eta.items():
                rows.append({
                    "architecture": arch_name,
                    "activation": act_name,
                    "optimizer": opt_name,
                    "eta": float(eta),
                    "mse": float(mse),
                })

df = pd.DataFrame(rows).sort_values(["architecture", "activation", "optimizer", "eta"]).reset_index(drop=True)


Top k runs (ranked by lowest MSE): 

In [ ]:
k = 10
top_k_overall = df.nsmallest(k, "mse").reset_index(drop=True)
print(top_k_overall)


                     architecture activation optimizer    eta       mse
0       2_Hidden_Layers (50, 100)       RELU  RMS_prop  0.001  0.010511
1       2_Hidden_Layers (50, 100)      LRELU  RMS_prop  0.001  0.010567
2       2_Hidden_Layers (50, 100)      LRELU      Adam  0.010  0.010695
3       2_Hidden_Layers (50, 100)       RELU      Adam  0.010  0.010726
4       2_Hidden_Layers (50, 100)      LRELU      Adam  0.001  0.011085
5  3_Hidden_Layers (50, 100, 200)       RELU  RMS_prop  0.001  0.011116
6  3_Hidden_Layers (50, 100, 200)      LRELU  RMS_prop  0.001  0.011135
7  3_Hidden_Layers (50, 100, 200)       RELU      Adam  0.001  0.011190
8  3_Hidden_Layers (50, 100, 200)      LRELU      Adam  0.001  0.011198
9       2_Hidden_Layers (50, 100)       RELU  RMS_prop  0.010  0.011230


In [ ]:
best_of_topk = top_k_overall.nsmallest(1, "mse").reset_index(drop=True)
b = best_of_topk.iloc[0]
print(f"{b['architecture']} | {b['activation']} | {b['optimizer']} | η={b['eta']:.1e} -> MSE={b['mse']:.6f}")


2_Hidden_Layers (50, 100) | RELU | RMS_prop | η=1.0e-03 -> MSE=0.010511


In [ ]:
# Map from activation label in your tables to actual activation functions
activation_map = {
    "Sigmoid": sigmoid,
    "RELU": RELU,
    "LReLU": LRELU,
    "LRELU": LRELU,   # in case of inconsistent capitalization
}

def build_nn_for_row(row):
    """Rebuild the FFNN corresponding to one row in best/top-k tables."""
    arch_name = row["architecture"]
    act_name  = row["activation"]
    
    # Architecture: use your existing dictionary
    layer_output_sizes = tuple(architectures_to_sweep[arch_name])
    
    # Hidden activations + identity output (regression)
    h = activation_map[act_name]
    n_hidden_layers = len(layer_output_sizes) - 1
    activation_funcs = [h]*n_hidden_layers + [identity]
    activation_ders  = [derivate(f) for f in activation_funcs]

    nn = FFNN(
        network_input_size=X_train_scaled.shape[1],
        layer_output_sizes=layer_output_sizes,
        activation_funcs=activation_funcs,
        activation_ders=activation_ders,
        cost_fun=CostOLS,
        cost_der=dCostOLS,
        seed=42,
    )
    nn.reset_weights()
    return nn

def make_scheduler(opt_name, eta):
    """Create the right optimizer from your optimizers_to_sweep setup."""
    opt_class, fixed = optimizers_to_sweep[opt_name]
    kwargs = {"eta": float(eta)}
    kwargs.update(fixed)
    return opt_class(**kwargs)

def train_eval(row, eta=None, l1=0.0, l2=0.0, lam=0.0, epochs=RETRAIN_EPOCHS):
    """
    Retrain one config described by `row` and return test MSE.
    Uses the architecture, activation, optimizer, and eta from the row,
    unless eta is explicitly overridden.
    """
    if eta is None:
        eta = float(row["eta"])
    nn = build_nn_for_row(row)
    scheduler = make_scheduler(row["optimizer"], eta)

    nn.fit(
        X=X_train_scaled,
        t=y_train,
        scheduler=scheduler,
        batches=32,
        epochs=int(epochs),
        lam=float(lam),  # keep lam separate from explicit l2
        l1=float(l1),
        l2=float(l2),
    )

    y_pred = nn.predict(X_fixed_test_scaled).ravel()
    return float(mean_squared_error(y_fixed_test.ravel(), y_pred))


In [ ]:
retrain_epochs = 500

def retrain_table(df_in, repeats=1):
    # Sanity checks
    needed = {"architecture","activation","optimizer","eta","mse"}
    missing = needed - set(df_in.columns)
    if missing:
        raise ValueError(f"Missing columns in df_in: {missing}")

    rows = []
    for i, r in df_in.iterrows():
        arch = r["architecture"]
        act  = r["activation"]
        opt  = r["optimizer"]
        eta  = float(r["eta"])
        old  = float(r["mse"])

        if repeats == 1:
            new_mse = _train_once(arch, act, opt, eta, reg_type)
        else:
            vals = [_train_once(arch, act, opt, eta, reg_type) for _ in range(repeats)]
            new_mse = float(np.mean(vals))

        delta = new_mse - old

        print(f"{i+1:02d}. {arch} | {act} | {opt} | η={eta:.1e}  "
              f"-> old={old:.6f} | new={new_mse:.6f} | Δ={delta:+.6f}")

        rows.append({
            "architecture": arch, "activation": act, "optimizer": opt,
            "eta": eta,
            "old_mse": old, "new_mse": new_mse, "delta": delta,
            "epochs": retrain_epochs, "repeats": repeats,
        })

    out = pd.DataFrame(rows).sort_values("new_mse").reset_index(drop=True)
    print(f"\n=== Retrain summary ({reg_type}) — top {len(df_in)} ===")
    display(out.head(min(10, len(out))))
    return out

# Retrain Top-K L1 and L2 (where topk_l1 has column 'l1' and topk_l2 has 'lam')
retrained_topk = retrain_table(best_of_topk, repeats=1)


NameError: name '_train_once' is not defined